# Unified S2D Diagnostic Architecture

Unified synthesis pipeline for subseasonal-to-decadal (S2D) prediction verification across multi-model ensembles.
This notebook implements the master synthesis workflow uniting:
1. **Branch 0 — Standardized Product Bundles**: Ingestion and validation of upstream diagnostic bundles produced across notebooks .
2. **Branch A — Field Drift**: Global and regional drift metrics (^{obs}, D^{obs}, J$).
3. **Branch B — Coupled Physical Consistency**: Inter-variable relationships (flux partitioning, coupling slopes, apparent residual).
4. **Branch C — Model Attractor**: Trajectory and distance analysis relative to an independent historical climatology ({att}$).
5. **Branch D — IC-to-Drift Attribution**: Spatial alignment and across-start attribution evaluating the role of initial condition differences.


In [ ]:
import os
import sys
from pathlib import Path

_repo_override = os.environ.get("ESP_LAB_REPO_ROOT")
_repo_candidates = (
    [Path(_repo_override).expanduser().resolve()]
    if _repo_override
    else [Path.cwd().resolve(), *Path.cwd().resolve().parents]
)
REPO_ROOT = next((p for p in _repo_candidates if (p / "workflows" / "diagnostics" / "unified").is_dir()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Start Jupyter inside ESP-Lab or set ESP_LAB_REPO_ROOT to its checkout.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

%load_ext autoreload
%autoreload 2
%matplotlib inline

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import dask
from IPython.display import display

ENVIRONMENT_SHARE = Path(sys.prefix) / 'share'
os.environ.setdefault('PROJ_DATA', str(ENVIRONMENT_SHARE / 'proj'))
os.environ.setdefault('GDAL_DATA', str(ENVIRONMENT_SHARE / 'gdal'))

from workflows.diagnostics.unified.config import (
    DEFAULT_OUTPUT_ROOT,
    READINESS_FLAGS,
    EXPERIMENTS,
    INIT_YEARS,
    INIT_MONTHS,
    DAILY_WINDOWS,
    MONTHLY_WINDOWS,
)


In [ ]:
print("xarray:", xr.__version__)
print("dask:", dask.__version__)


In [ ]:
from esp_lab.utils.notebook_resources import restart_notebook_cluster, close_notebook_resources
from esp_lab.utils.dask_util import get_cluster_client, DaskConfig

machine_env = os.environ.get("CLUSTER_TYPE", "local")
dask_cfg = DaskConfig(
    cluster_type=machine_env,
    workers=16,
    cores=4,
    memory="16GB",
    walltime="02:00:00",
    queue=None,
    project=None,
)

cluster, client, workflow_resources = restart_notebook_cluster(
    globals(), lambda: get_cluster_client(dask_cfg)
)
print(client)


## User Control Panel


In [ ]:
# =========================================================================
# USER CONTROL PANEL
# =========================================================================
FREQUENCY = "monthly"       # "daily" or "monthly"
VARIABLE = "TREFHT"
OUTPUT_ROOT = Path(DEFAULT_OUTPUT_ROOT) / "unified_synthesis"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
BASELINE_LEAD = 1
PRODUCT_BUNDLES = []        # Paths to JSON bundles from upstream notebooks (5d--5j)

RUN_BRANCH_A = True         # Field drift
RUN_BRANCH_B = True         # Physical consistency
RUN_BRANCH_C = True         # Model attractor
RUN_BRANCH_D = False        # IC attribution (requires paired IC diff)
# =========================================================================


## 1. Inventory & Readiness Check


In [ ]:
%%time
from workflows.diagnostics.unified.inventory import run_inventory

inv_summary = run_inventory(output_root=OUTPUT_ROOT, verbose=True)
print("Readiness summary:", inv_summary.get("status", "completed"))


## 2. Load Input & Reference Fields


In [ ]:
%%time
from workflows.diagnostics.unified.references import (
    build_observation_reference,
    build_e3sm_historical_climatology,
)
from esp_lab.diagnostics.monthly_io import load_campaign_field

# Load or generate synthetic fields for pipeline execution
try:
    model_field = load_campaign_field("JRA55_FOSIRL", VARIABLE, season="may", frequency=FREQUENCY)
    obs_ref = build_observation_reference(VARIABLE, frequency=FREQUENCY)
    e3sm_clim = build_e3sm_historical_climatology(VARIABLE, frequency=FREQUENCY)
    print("Successfully loaded model and reference fields from archive.")
except Exception as err:
    print(f"Data archive loading note: {err}")
    print("Generating demo grid arrays for unified execution...")
    lats = np.linspace(-90, 90, 180)
    lons = np.linspace(0, 360, 360, endpoint=False)
    leads = np.arange(1, 13)
    years = np.arange(1980, 1985)
    coords = {"Y": years, "L": leads, "lat": lats, "lon": lons}
    shape = (len(years), len(leads), len(lats), len(lons))
    
    np.random.seed(42)
    model_field = xr.DataArray(
        np.random.normal(288.0, 5.0, size=shape).astype(np.float32),
        coords=coords, dims=["Y", "L", "lat", "lon"], attrs={"units": "K"}
    )
    obs_ref = xr.DataArray(
        np.random.normal(287.5, 4.8, size=(len(leads), len(lats), len(lons))).astype(np.float32),
        coords={"L": leads, "lat": lats, "lon": lons}, dims=["L", "lat", "lon"], attrs={"units": "K"}
    )
    e3sm_clim = xr.DataArray(
        np.random.normal(287.8, 4.9, size=(len(leads), len(lats), len(lons))).astype(np.float32),
        coords={"L": leads, "lat": lats, "lon": lons}, dims=["L", "lat", "lon"], attrs={"units": "K"}
    )
    print("Demo fields generated with shape:", model_field.shape)


## 3. Run Unified Master Pipeline


In [ ]:
%%time
from workflows.diagnostics.unified.run_unified import run as run_unified_pipeline

results = run_unified_pipeline(
    frequency=FREQUENCY,
    variable=VARIABLE,
    output_root=OUTPUT_ROOT,
    model_field=model_field,
    obs_ref=obs_ref,
    e3sm_clim=e3sm_clim,
    product_bundles=PRODUCT_BUNDLES,
    baseline_lead=BASELINE_LEAD,
    verbose=True,
)
print("Unified pipeline execution keys:", list(results.keys()))


## 4. Branch A — Field Drift Diagnostics


In [ ]:
if "field_drift" in results:
    drift_res = results["field_drift"]
    print("Field drift components:", list(drift_res.keys()))
    if "distance_change_obs" in drift_res:
        d_obs = drift_res["distance_change_obs"]
        print(f"D_obs mean: {float(d_obs.mean()):.4f} K, shape: {d_obs.shape}")
else:
    print("Branch A skipped.")


## 5. Branch B — Physical Consistency


In [ ]:
if "physical_consistency" in results:
    phys_res = results["physical_consistency"]
    print("Physical consistency metrics:", list(phys_res.keys()))
else:
    print("Branch B direct call skipped (synthesized via product bundles).")


## 6. Branch C — Model Attractor Diagnostics


In [ ]:
if "model_attractor" in results:
    att_res = results["model_attractor"]
    print("Model attractor diagnostic outputs:", list(att_res.keys()))
    for k, v in att_res.items():
        if isinstance(v, xr.DataArray):
            print(f"  {k}: mean={float(v.mean(skipna=True)):.4f}, shape={v.shape}")
else:
    print("Branch C skipped.")


## 7. Branch D — IC-to-Drift Attribution


In [ ]:
if "ic_drift_attribution" in results:
    attr_res = results["ic_drift_attribution"]
    print("Attribution results:", list(attr_res.keys()))
else:
    print("Branch D skipped (requires paired initial condition perturbation dataset).")


## 8. Synthesis Product Table


In [ ]:
if "synthesis_product_table" in results:
    table = results["synthesis_product_table"]
    print(f"Synthesized product table contains {len(table)} rows:")
    display(table.head(10))
else:
    print("No synthesized product table written.")


## 9. Validation & Teardown


In [ ]:
assert results, "Unified pipeline returned empty results"
assert "inventory" in results, "Inventory check missing"
if "field_drift" in results:
    assert "distance_change_obs" in results["field_drift"]

# Clean up dask resources
close_notebook_resources(workflow_resources)
print("Unified diagnostics workflow successfully completed and cluster resources closed.")
